In [1]:
import os
import sys
from pathlib import Path

if Path.cwd().name == 'notebooks':
    os.chdir('..')
project_root = Path.cwd().resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

In [2]:
import torch
from qiskit import transpile
from torch.utils.data import DataLoader

from modules.common import get_device, load_pkl, set_seed
from modules.symbolic.text_processor import TextProcessor
from modules.data_pipeline.retrieval import BaseDatasetRetriever
from modules.data_pipeline.embeddings import ImgStream, embed_images
from modules.data_pipeline.datasets import (
    SVODataset, SwapDataset, build_swap_set, svo_collate_fn,
    text_choice_collate_fn,
)
from modules.compilation.classical.tensor_network import TNCompiler
from modules.compilation.classical.decompositions import MPS
from modules.models.text.einsum_classical import TNModel
from modules.models.vision.clip import FrozenCLIP
from modules.models.fusion.criteria import InfoNCE

set_seed(0)
device = get_device()
data_root = Path('data')
processed_root = data_root / 'svo-probes' / 'processed'
all_path = processed_root / 'all.pkl'
image_path = processed_root / 'image_embeddings.pt'
batch_size = 256
num_epochs = 1

In [3]:
if not all_path.exists() or not image_path.exists():
    retriever = BaseDatasetRetriever('svo-probes', data_root=data_root)
    retriever.retrieve()
    TextProcessor(
        parser_path=os.getenv('BOBCAT_PATH', './bobcat'),
        batch_size=64,
        device=str(device),
    ).text2diagram(
        path=processed_root,
        dataset=retriever.data,
        text_labels=['corrected_sentence'],
    )
    embed_images(ImgStream(retriever.images_zip, file_type='zip'), image_path, device=str(device))

all_df = load_pkl(all_path)
train_df = all_df[all_df['legacy_split'] == 'train'].copy().reset_index(drop=True)
test_df = all_df[all_df['legacy_split'] == 'test'].copy().reset_index(drop=True)

In [4]:
tn_compiler = TNCompiler(
    obmap={'n': 4, 's': 4, 'p': 4, 'out': 64},
    decomp_fn=MPS(bond_dim=2, max_order=2),
    output_mode='head',
)
train_df = tn_compiler.compile_dataset(train_df)
test_df = tn_compiler.compile_dataset(test_df)

text_model = TNModel(out_dim=64)
text_model.from_symbols(
    train_df['corrected_sentence_symbols'].tolist() + 
    test_df['corrected_sentence_symbols'].tolist()
)
text_model = text_model.to(device)
image_model = FrozenCLIP(out_dim=64).to(device)
loss_fn = InfoNCE()
optimizer = torch.optim.Adam(text_model.parameters(), lr=1e-3)
train_loader = DataLoader(
    SVODataset(train_df, image_path),
    batch_size=batch_size,
    shuffle=True,
    collate_fn=svo_collate_fn,
)

100%|██████████| 5961/5961 [00:00<00:00, 379126.36it/s]


In [5]:
for epoch in range(num_epochs):
    text_model.train()
    image_model.eval()
    total_loss = 0.0
    for batch in train_loader:
        image_emb = image_model(batch['pos_image'].to(device))
        text_emb = text_model(batch['caption'])
        loss = loss_fn(text_emb, image_emb)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(batch['caption'])
    print(f'TN epoch {epoch + 1}/{num_epochs} | loss: {total_loss / len(train_loader.dataset):.4f}')


TN epoch 1/1 | loss: 9.5481


In [6]:
from modules.models.fusion.evaluator import eval_img_choice

loader = DataLoader(
    SVODataset(test_df, image_path),
    batch_size=batch_size,
    shuffle=False,
    collate_fn=svo_collate_fn,
)
tn_image_accuracy = eval_img_choice(image_model, text_model, loader, device)
print(f'TN image-choice accuracy: {tn_image_accuracy:.4f}')

TN image-choice accuracy: 0.5357


In [7]:
from modules.models.fusion.evaluator import eval_txt_choice

swap_df = tn_compiler.compile_dataset(build_swap_set(test_df))
swap_loader = DataLoader(
    SwapDataset(swap_df, image_path),
    batch_size=batch_size,
    shuffle=False,
    collate_fn=text_choice_collate_fn,
)
tn_swap_accuracy = eval_txt_choice(image_model, text_model, swap_loader, device)
print(f'TN swap caption-choice accuracy: {tn_swap_accuracy:.4f}')

Swap set: {'total': 1525, 'not_two_nouns': 33, 'same_noun': 0, 'symmetric': 13} -> kept 1479


100%|██████████| 1479/1479 [00:00<00:00, 51945.87it/s]


TN swap caption-choice accuracy: 0.5274


In [8]:
from modules.compilation.quantum.ansatz import BaseAnsatz

class BareAnsatz(BaseAnsatz):
    def ansatz(self, current_wires, base_symbol):
        new_indices, new_tensors = [], []
        for layer in range(self.layers):
            for wire in range(len(current_wires)):
                nxt = self.get_char()
                new_indices.append(current_wires[wire] + nxt)
                new_tensors.append({'name': f'{base_symbol}_l{layer}_q{wire}', 'op_type': 'Ry'})
                current_wires[wire] = nxt
            for wire in range(len(current_wires) - 1):
                self._cx(current_wires, wire, wire + 1, new_indices, new_tensors)
        return current_wires, new_indices, new_tensors

vqc_qubits = 1
vqc_ansatz = BareAnsatz(
    obmap={'n': 1, 's': 1, 'p': 1, 'out': vqc_qubits},
    layers=1,
)

In [9]:
from modules.models.text.einsum_quantum import VQCModel
from modules.models.vision.quantum_map import QuantumFeatureMap
from modules.models.fusion.criteria import QInfoNCE

vqc_train_df = vqc_ansatz.compile_dataset(train_df.copy())
vqc_test_df = vqc_ansatz.compile_dataset(test_df.copy())
vqc_text_model = VQCModel(out_q=vqc_qubits)
vqc_text_model.from_symbols(
    vqc_train_df['corrected_sentence_symbols'].tolist() + 
    vqc_test_df['corrected_sentence_symbols'].tolist()
)
vqc_text_model = vqc_text_model.to(device)
vqc_image_model = QuantumFeatureMap(
    k=vqc_qubits, layers=2, vdim=512, batch_size=batch_size,
).to(device)
vqc_loss_fn = QInfoNCE()
vqc_optimizer = torch.optim.Adam(vqc_text_model.parameters(), lr=1e-3)
vqc_train_loader = DataLoader(
    SVODataset(vqc_train_df, image_path),
    batch_size=batch_size,
    shuffle=True,
    collate_fn=svo_collate_fn,
)

100%|██████████| 5961/5961 [00:00<00:00, 140805.37it/s]


In [10]:
for epoch in range(num_epochs):
    vqc_text_model.train()
    vqc_image_model.eval()
    total_loss = 0.0
    for batch in vqc_train_loader:
        image_emb = vqc_image_model(batch['pos_image'].to(device))
        text_emb = vqc_text_model(batch['caption'])
        loss = vqc_loss_fn(text_emb, image_emb)
        vqc_optimizer.zero_grad(set_to_none=True)
        loss.backward()
        vqc_optimizer.step()
        total_loss += loss.item() * len(batch['caption'])
    print(f'VQC epoch {epoch + 1}/{num_epochs} | loss: {total_loss / len(vqc_train_loader.dataset):.4f}')

VQC epoch 1/1 | loss: 9.5782


In [11]:
vqc_test_loader = DataLoader(
    SVODataset(vqc_test_df, image_path),
    batch_size=batch_size,
    shuffle=False,
    collate_fn=svo_collate_fn,
)
vqc_image_accuracy = eval_img_choice(vqc_image_model, vqc_text_model, vqc_test_loader, device)
print(f'VQC perfect-simulation image-choice accuracy: {vqc_image_accuracy:.4f}')

VQC perfect-simulation image-choice accuracy: 0.4898


In [12]:
from modules.compilation.quantum.circuits import (
    compile_image_circuits,
    compile_text_circuits,
    postselect_output_counts,
)

svo_swap_df = vqc_ansatz.compile_dataset(
    build_swap_set(test_df).head(95).reset_index(drop=True)
)
vqc_test_dataset = SVODataset(svo_swap_df, image_path)
text_recipes = [vqc_test_dataset[i]['caption'] for i in range(len(vqc_test_dataset))]
positive_images = [vqc_test_dataset[i]['pos_image'] for i in range(len(vqc_test_dataset))]
negative_images = [vqc_test_dataset[i]['neg_image'] for i in range(len(vqc_test_dataset))]
text_circuits = compile_text_circuits(
    text_recipes, vqc_text_model, vqc_qubits,
)
positive_circuits = compile_image_circuits(
    positive_images, vqc_image_model, vqc_qubits,
)
negative_circuits = compile_image_circuits(
    negative_images, vqc_image_model, vqc_qubits,
)
paired_circuits = list(zip(text_circuits, positive_circuits, negative_circuits))
if not paired_circuits:
    raise ValueError('No aligned circuit triples were compiled')
text_circuits, positive_circuits, negative_circuits = map(list, zip(*paired_circuits))
print(f'Compiled {len(text_circuits)} aligned circuit triples')


Swap set: {'total': 1525, 'not_two_nouns': 33, 'same_noun': 0, 'symmetric': 13} -> kept 1479


Compiling text circuits: 100%|██████████| 95/95 [00:00<00:00, 2418.96circuit/s]


 Compiled 95 text circuits | Dropped 0 wider than None qubits


Compiling image circuits: 100%|██████████| 95/95 [00:00<00:00, 779.82circuit/s]


 Compiled 95 image circuits | Dropped 0 wider than None qubits


Compiling image circuits: 100%|██████████| 95/95 [00:00<00:00, 792.86circuit/s]

 Compiled 95 image circuits | Dropped 0 wider than None qubits
Compiled 95 aligned circuit triples


In [13]:
from modules.compilation.quantum.backends import BackendManager, fake_backend

emulation_provider = os.getenv('EMULATION_PROVIDER', 'iqm')
profile_candidates = {
    'iqm': ['adonis', 'aphrodite', 'apollo', 'garnet'],
    'ibm': ['miami'],
}[emulation_provider]
emulation_profile = os.getenv('EMULATION_PROFILE')
if emulation_profile is None:
    emulation_profile = max(
        profile_candidates,
        key=lambda profile: fake_backend(profile, emulation_provider).num_qubits,
    )
hardware_profile = fake_backend(emulation_profile, emulation_provider)
backend_qubits = hardware_profile.num_qubits
paired_circuits = [
    triple for triple in zip(text_circuits, positive_circuits, negative_circuits)
    if max(circuit.num_qubits for circuit in triple) <= backend_qubits
]
if not paired_circuits:
    raise ValueError(f'No circuit triples fit the {backend_qubits}-qubit backend')
text_circuits, positive_circuits, negative_circuits = map(list, zip(*paired_circuits))
qlimit = max(
    circuit.num_qubits
    for circuits in (text_circuits, positive_circuits, negative_circuits)
    for circuit in circuits
)
print(f'Using {emulation_profile} with {backend_qubits} qubits; qlimit={qlimit}')

ideal_manager = BackendManager(
    provider=emulation_provider,
    profile=emulation_profile,
    method='automatic',
    hardware_profile=hardware_profile,
    qlimit=qlimit,
    noisy=False,
)

noisy_manager = BackendManager(
    provider=emulation_provider,
    profile=emulation_profile,
    method='automatic',
    hardware_profile=hardware_profile,
    qlimit=qlimit,
    noisy=True,
)

POSTSELECTION_SHOTS = 1024

def measured(circuit):
    measured_circuit = circuit.copy()
    measured_circuit.measure_all()
    return measured_circuit

from tqdm.auto import tqdm

def run_counts(circuits, manager, label, shots=POSTSELECTION_SHOTS, batch_size=4):
    compiled = manager.transpile([measured(circuit) for circuit in circuits])
    counts = []
    for start in tqdm(
        range(0, len(compiled), batch_size),
        desc=f'{label} {"noisy" if manager.noisy else "noiseless"}',
        unit='batch',
    ):
        chunk = compiled[start:start + batch_size]
        result = manager.backend.run(chunk, shots=shots).result()
        counts.extend(
            postselect_output_counts(result.get_counts(i), circuits[start + i])
            for i in range(len(chunk))
        )
    return counts

def sampled_overlap(left_counts, right_counts):
    left_total = sum(left_counts.values())
    right_total = sum(right_counts.values())
    if not left_total or not right_total:
        raise ValueError('Cannot compute sampled overlap without accepted shots')
    keys = set(left_counts) | set(right_counts)
    left = torch.tensor([left_counts.get(key, 0) / left_total for key in keys])
    right = torch.tensor([right_counts.get(key, 0) / right_total for key in keys])
    return float(torch.sqrt(left * right).sum().square())

def evaluate_count_triples(text_counts, positive_counts, negative_counts, label):
    triples = [
        (text, positive, negative)
        for text, positive, negative in zip(text_counts, positive_counts, negative_counts)
        if text and positive and negative
    ]
    total = len(text_counts)
    if not triples:
        raise ValueError(f'No circuit triples survived post-selection for {label} (0/{total})')
    print(f'{label}: post-selection retained {len(triples)}/{total} circuit triples')
    positive_scores = [sampled_overlap(text, positive) for text, positive, _ in triples]
    negative_scores = [sampled_overlap(text, negative) for text, _, negative in triples]
    return (torch.tensor(positive_scores) > torch.tensor(negative_scores)).float().mean().item()


Using aphrodite with 54 qubits; qlimit=19
 Simulation: iqm profile IQMFakeAphroditeBackend (19 qubits), Device CPU, Method automatic, Noise Disabled
 Simulation: iqm profile IQMFakeAphroditeBackend (19 qubits), Device CPU, Method automatic, Noise Enabled


In [14]:
def evaluate_emulation(manager, shots=POSTSELECTION_SHOTS):
    text_counts = run_counts(text_circuits, manager, 'text', shots)
    positive_counts = run_counts(positive_circuits, manager, 'positive', shots)
    negative_counts = run_counts(negative_circuits, manager, 'negative', shots)
    return evaluate_count_triples(text_counts, positive_counts, negative_counts, 'emulation')

noiseless_emulation_accuracy = evaluate_emulation(ideal_manager)
noisy_emulation_accuracy = evaluate_emulation(noisy_manager)
print(f'{emulation_provider} noiseless-emulation image-choice accuracy: {noiseless_emulation_accuracy:.4f}')
print(f'{emulation_provider} noisy-emulation image-choice accuracy: {noisy_emulation_accuracy:.4f}')


Transpiling: 100%|██████████| 1/1 [00:00<00:00,  3.26batch/s]


text noiseless:   0%|          | 0/24 [00:00<?, ?batch/s]

Transpiling: 100%|██████████| 1/1 [00:00<00:00, 13.10batch/s]


positive noiseless:   0%|          | 0/24 [00:00<?, ?batch/s]

Transpiling: 100%|██████████| 1/1 [00:00<00:00, 13.16batch/s]


negative noiseless:   0%|          | 0/24 [00:00<?, ?batch/s]

... (4 outputs left out)

positive noisy:   0%|          | 0/24 [00:00<?, ?batch/s]

Transpiling: 100%|██████████| 1/1 [00:00<00:00, 10.60batch/s]


negative noisy:   0%|          | 0/24 [00:00<?, ?batch/s]

emulation: post-selection retained 6/95 circuit triples
iqm noiseless-emulation image-choice accuracy: 0.6250
iqm noisy-emulation image-choice accuracy: 0.6667


In [18]:
RUN_HARDWARE = False
IQM_URL = os.getenv('IQM_URL')
IQM_TOKEN = os.getenv('IQM_TOKEN')
IQM_DEVICE = os.getenv('IQM_DEVICE', 'Garnet')
IBM_BACKEND = os.getenv('IBM_BACKEND')
IBM_CHANNEL = os.getenv('IBM_CHANNEL', 'ibm_quantum_platform')

In [19]:
if RUN_HARDWARE and IQM_URL and IQM_TOKEN:
    from iqm.qiskit_iqm import IQMProvider

    iqm_provider = IQMProvider(IQM_URL, IQM_TOKEN)
    iqm_backend = iqm_provider.get_backend(IQM_DEVICE)
    iqm_text_counts, iqm_positive_counts, iqm_negative_counts = [], [], []
    for text, pos, neg in zip(text_circuits, positive_circuits, negative_circuits):
        text_job = iqm_backend.run(transpile(measured(text), iqm_backend), shots=POSTSELECTION_SHOTS)
        pos_job = iqm_backend.run(transpile(measured(pos), iqm_backend), shots=POSTSELECTION_SHOTS)
        neg_job = iqm_backend.run(transpile(measured(neg), iqm_backend), shots=POSTSELECTION_SHOTS)
        iqm_text_counts.append(postselect_output_counts(text_job.result().get_counts(), text))
        iqm_positive_counts.append(postselect_output_counts(pos_job.result().get_counts(), pos))
        iqm_negative_counts.append(postselect_output_counts(neg_job.result().get_counts(), neg))
    iqm_hardware_accuracy = evaluate_count_triples(
        iqm_text_counts, iqm_positive_counts, iqm_negative_counts, 'IQM hardware'
    )

In [20]:
if RUN_HARDWARE and IBM_BACKEND:
    from qiskit_ibm_runtime import QiskitRuntimeService

    ibm_service = QiskitRuntimeService(channel=IBM_CHANNEL)
    ibm_backend = ibm_service.backend(IBM_BACKEND)
    ibm_text_counts, ibm_positive_counts, ibm_negative_counts = [], [], []
    for text, pos, neg in zip(text_circuits, positive_circuits, negative_circuits):
        text_job = ibm_backend.run(transpile(measured(text), ibm_backend), shots=POSTSELECTION_SHOTS)
        pos_job = ibm_backend.run(transpile(measured(pos), ibm_backend), shots=POSTSELECTION_SHOTS)
        neg_job = ibm_backend.run(transpile(measured(neg), ibm_backend), shots=POSTSELECTION_SHOTS)
        ibm_text_counts.append(postselect_output_counts(text_job.result().get_counts(), text))
        ibm_positive_counts.append(postselect_output_counts(pos_job.result().get_counts(), pos))
        ibm_negative_counts.append(postselect_output_counts(neg_job.result().get_counts(), neg))
    ibm_hardware_accuracy = evaluate_count_triples(
        ibm_text_counts, ibm_positive_counts, ibm_negative_counts, 'IBM hardware'
    )
